In [ ]:
el = [
    0,
    5,
    7,
    8,
    9,
    16,
    26,
    29,
    30,
    33,
    39,
    40,
    41,
    44,
    45,
    46,
    49,
    51,
    52,
    54,
    55,
    58,
    59,
    61,
]
print(el[2], el[23])  # subj 3

In [ ]:
el = [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    20,
    21,
    22,
    24,
    27,
    28,
    29,
]
print(el[8], el[9])  # subj 6

In [ ]:
import matplotlib.pyplot as plt
from scipy.io import loadmat
import numpy as np
from scipy import signal

ieeg = loadmat("../NonLinearityData/iEEG/iEEG_ID03.mat")["EEG"]

In [ ]:
f_sample = 512

sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.sosfiltfilt(sos, ieeg[:, :])
beg, end = 10000, 12000  # 0,1843200

# The time series we used

In [ ]:
plt.figure(figsize=(20, 5))
x = np.arange(beg, end) / f_sample
for i in range(64):
    if i in [7, 61]:
        continue
    fact = max(np.abs(ieeg[61, beg:end])) / max(np.abs(ieeg[i, beg:end]))
    plt.plot(x, fact * ieeg[i, beg:end], "--", color="gray", lw=1, alpha=0.5)

fact = max(np.abs(ieeg[61, beg:end])) / max(np.abs(ieeg[7, beg:end]))
plt.plot(
    x, fact * ieeg[7, beg:end], lw=2, label=f"{fact:.3} " + r"$\times$ electrode 7"
)
plt.plot(x, ieeg[61, beg:end], "--", lw=2, label="electrode 61")
plt.xlabel("Time [s]")
plt.ylabel(r"$\propto$ V")
plt.xlim(beg / f_sample, end / f_sample)
plt.legend()
plt.show(block=True)

## Looking only at band $\delta$

In [ ]:
plt.figure(figsize=(20, 5))
x = np.arange(beg, end) / f_sample
for i in range(64):
    if i in [7, 61]:
        continue
    fact = max(np.abs(filtered[61, beg:end])) / max(np.abs(filtered[i, beg:end]))
    plt.plot(x, fact * filtered[i, beg:end], "--", color="gray", lw=1, alpha=0.5)
fact = max(np.abs(filtered[61, beg:end])) / max(np.abs(filtered[7, beg:end]))
plt.plot(
    x, 8 * filtered[7, beg:end], lw=2, label=f"{fact:.3} " + r"$\times$ electrode 7"
)
plt.plot(x, filtered[61, beg:end], "--", lw=2, label="electrode 61")
plt.xlabel("Time [s]")
plt.ylabel("V")
plt.xlim(beg / f_sample, end / f_sample)
plt.legend()
plt.show(block=True)

# Getting more hours

Excluding the ones containing seizures, ~1h every 16.

In [ ]:
import os
import urllib.request as ur
import io

baseURL = "http://ieeg-swez.ethz.ch/long-term_dataset/"
basePath = "/mnt/DATA/NonLinearMI/iEEG"
appendix = "info.mat"

for i in [3]:
    sub = f"ID{i:02}"
    fname = "_".join([sub, appendix])
    URL = os.path.join(baseURL, sub, fname)
    request_data = ur.urlopen(URL)
    file_like = io.BytesIO(request_data.read())
    mat = loadmat(file_like)
    bad_hours = set()
    for p in zip(mat["seizure_begin"], mat["seizure_end"]):
        for e in p:
            bad_hours.add(int(e[0] / 3600) + 1)
    if i in [15, 17]:
        bad_hours.add(0)
print(bad_hours)

In [ ]:
basePath = "../NonLinearityData/iEEG/"
sub = f"ID03"
i = 1
for j in range(11):
    data_fname = "_".join([sub, f"{i}h.mat"])
    mat_path = os.path.join(baseURL, sub, data_fname)
    print(i, mat_path)
    out_fname = f"iEEG_ID{3:02}_{i:03}h.mat"
    out_path = os.path.join(basePath, out_fname)
    if not os.path.isfile(out_path):
        print(ur.urlretrieve(mat_path, out_path)[0])
    i += 16
    if i % 7 == 0:
        i -= 1

In [ ]:
def moving_average(a, n=10):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1 :] / n


fs = 10
MA = 100
off, dur = 100000, 20000
beg, end = off, off + dur
x = np.arange(int(beg / f_sample * fs), int(end / f_sample * fs)) / fs
# x = np.arange(beg, end)/fs
i = 1
for j in range(11):
    tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_{i:03}h.mat")["EEG"]
    sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
    filtered = signal.resample(
        signal.sosfiltfilt(sos, tieeg[[7, 61], beg:end]).T, int(dur / f_sample * fs)
    ).T

    plt.figure(figsize=(20, 5))

    ratio = np.std(filtered[1, :]) / np.std(filtered[0, :])
    plt.plot(x, ratio * filtered[0, :], label=f"{ratio:.3} " + r"$\times$ electrode 7")
    plt.plot(x, filtered[1, :], "--", label="electrode 61")

    plt.xlabel("Time [s]")
    plt.ylabel(r"$\langle\Delta\phi\rangle$")
    plt.xlim((beg) / f_sample, end / f_sample)
    plt.title(f"{i}° hour")
    plt.legend()
    plt.show()
    i += 16
    if i % 7 == 0:
        i -= 1

In [ ]:
def moving_average(a, n=10):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1 :] / n


fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
# x = np.arange(beg, end)/fs
i = 1
for j in range(11):
    tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_{i:03}h.mat")["EEG"]
    sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
    filtered = signal.resample(
        signal.sosfiltfilt(sos, tieeg[[7, 61], :]).T, 3600 * fs
    ).T

    plt.figure(figsize=(20, 5))

    z1 = signal.hilbert(filtered[0, beg:end])
    z2 = signal.hilbert(filtered[1, beg:end])

    phadif = np.diff(np.unwrap(np.angle(z1)) - np.unwrap(np.angle(z2))) * fs
    plt.plot(x[MA:], moving_average(phadif, MA))  # np.abs(phadif)
    plt.plot([0, 3600], [np.pi] * 2, ":k")
    plt.plot([0, 3600], [-np.pi] * 2, ":k")
    plt.xlabel("Time [s]")
    plt.ylabel(r"$\langle\Delta\phi\rangle$")
    plt.xlim((beg + MA) / f_sample, end / f_sample)
    plt.title(f"{i}° hour")
    # plt.legend()
    plt.show()
    # plt.hist(moving_average(phadif[:16000], MA), bins="auto", density=True)
    # plt.hist(moving_average(phadif[16000:], MA), bins="auto", density=True, alpha=0.7)
    # plt.show()
    i += 16
    if i % 7 == 0:
        i -= 1

In [ ]:
def moving_average(a, n=10):
    ret = np.cumsum(a, dtype=float)
    ret[n:] = ret[n:] - ret[:-n]
    return ret[n - 1 :] / n


fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
# x = np.arange(beg, end)/fs
i = 1
for j in range(11):
    tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_{i:03}h.mat")["EEG"]
    sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
    filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[:, :]).T, 3600 * fs).T
    z = signal.hilbert(filtered)
    alpha = np.unwrap(np.angle(z))
    plt.figure(figsize=(20, 5))

    out = np.zeros((64, 64))
    for k in range(64):
        for l in range(k + 1, 64):
            phadif = np.diff(alpha[k, :] - alpha[l, :], prepend=0) * fs
            out[k, l] = np.min(np.std(np.reshape(phadif, (-1, 100)), 1))
    out += out.T
    np.fill_diagonal(out, np.nan)
    plt.imshow(out)
    plt.colorbar()
    plt.title(f"{i}° hour")
    # plt.legend()
    plt.show()
    # plt.hist(moving_average(phadif[:16000], MA), bins="auto", density=True)
    # plt.hist(moving_average(phadif[16000:], MA), bins="auto", density=True, alpha=0.7)
    # plt.show()
    i += 16
    if i % 7 == 0:
        i -= 1

# Deepening in the 1st hour

In [ ]:
candi = [7, 48, 56, 57, 58, 61, 62]

fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_001h.mat")["EEG"]
sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[candi, :]).T, 3600 * fs).T
z = signal.hilbert(filtered)
alpha = np.unwrap(np.angle(z))

for k in range(len(candi)):
    for l in range(k + 1, len(candi)):
        phadif = np.diff(alpha[k, :] - alpha[l, :], prepend=0) * fs
        dev = np.std(np.reshape(phadif, (-1, 100)), 1)
        plt.figure(figsize=(10, 3))
        plt.plot(x[::100], dev)
        plt.title(f"{candi[k]} - {candi[l]}")
        plt.plot([0, 3600], [8, 8], ":r")
        plt.show()

In [ ]:
from mienc.support import normalise

candi = [7, 48, 56, 57, 58, 61, 62]
inner = [7, 48, 56, 58, 61]
fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_001h.mat")["EEG"]
sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[candi, :]).T, 3600 * fs).T[
    :, 4400:5000
]
for k in range(len(candi)):
    filtered[k, :] = normalise(filtered[k, :])
fig, ax = plt.subplots(5, 4, figsize=(18, 18))
n = 0
for k in range(len(candi)):
    for l in range(k + 1, len(candi)):
        ax[n // 4, n % 4].scatter(
            filtered[k, :], filtered[l, :], c=np.arange(600), alpha=0.4
        )
        ax[n // 4, n % 4].set_title(f"{candi[k]} - {candi[l]}")
        if candi[k] in inner and candi[l] in inner:
            ax[n // 4, n % 4].scatter([0], [0], marker="*", color="r", alpha=0.8)
        ax[n // 4, n % 4].set_xticks([])
        ax[n // 4, n % 4].set_yticks([])
        ax[n // 4, n % 4].set_aspect("1")
        # ax[n//5,n%5].plot([0,3600], [8,8], ":r")
        n += 1
        if n == 20:
            break
    if n == 20:
        break
plt.show()

In [ ]:
import pandas as pd

candi = [7, 48, 56, 57, 58, 61, 62]
inner = [7, 48, 56, 58, 61]
fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_001h.mat")["EEG"]
sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[candi, :]).T, 3600 * fs).T
asas = np.argsort(np.argsort(filtered[:, 4400:5000], axis=1), axis=1)
fig, ax = plt.subplots(5, 4, figsize=(18, 18))
n = 0
for k in range(len(candi)):
    for l in range(k + 1, len(candi)):
        ax[n // 4, n % 4].hist2d(asas[k, :], asas[l, :], bins=9)
        ax[n // 4, n % 4].set_title(f"{candi[k]} - {candi[l]}")
        if candi[k] in inner and candi[l] in inner:
            ax[n // 4, n % 4].scatter([300], [300], marker="*", color="r", alpha=0.8)
        ax[n // 4, n % 4].set_xticks([])
        ax[n // 4, n % 4].set_yticks([])
        ax[n // 4, n % 4].set_aspect("1")
        # ax[n//5,n%5].plot([0,3600], [8,8], ":r")
        n += 1
        if n == 20:
            break
    if n == 20:
        break
plt.show()

In [ ]:
n

# Deepening in the 111th hour

In [ ]:
candi = [5, 6, 7, 13, 14, 15]  # [14,15,17,18,19,26,27]

fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_143h.mat")["EEG"]
sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[candi, :]).T, 3600 * fs).T
z = signal.hilbert(filtered)
alpha = np.unwrap(np.angle(z))

for k in range(len(candi)):
    for l in range(k + 1, len(candi)):
        phadif = np.diff(alpha[k, :] - alpha[l, :], prepend=0) * fs
        dev = np.std(np.reshape(phadif, (-1, 100)), 1)
        plt.figure(figsize=(10, 3))
        plt.plot(x[::100], dev)
        plt.title(f"{candi[k]} - {candi[l]}")
        plt.plot([0, 3600], [8, 8], ":r")
        plt.show()

In [ ]:
candi = [5, 6, 7, 13, 14, 15]  # [14,15,17,18,19,26,27]
inner = [7, 48, 56, 58, 61]
fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_001h.mat")["EEG"]
sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[candi, :]).T, 3600 * fs).T[
    :, 22000:22600
]
for k in range(len(candi)):
    filtered[k, :] = normalise(filtered[k, :])

fig, ax = plt.subplots(5, 4, figsize=(18, 18))
n = 0
for k in range(len(candi)):
    for l in range(k + 1, len(candi)):
        fact = np.std(filtered[l, :]) / np.std(filtered[k, :])
        ax[n // 4, n % 4].scatter(
            fact * filtered[k, :], filtered[l, :], c=np.arange(600), alpha=0.4
        )
        ax[n // 4, n % 4].set_title(f"{candi[k]} - {candi[l]}")
        if candi[k] in inner and candi[l] in inner:
            ax[n // 4, n % 4].scatter([0], [0], marker="*", color="r", alpha=0.8)
        ax[n // 4, n % 4].set_xticks([])
        ax[n // 4, n % 4].set_yticks([])
        ax[n // 4, n % 4].set_aspect("1")
        # ax[n//5,n%5].plot([0,3600], [8,8], ":r")
        n += 1
        if n == 20:
            break
    if n == 20:
        break
plt.show()

In [ ]:
candi = [5, 6, 7, 13, 14, 15]  # [14,15,17,18,19,26,27]
inner = [7, 48, 56, 58, 61]
fs = 10
MA = 100
off, dur = 80000, 30000
beg, end = 0, 1843200  # off, off+dur
x = np.arange(beg, end / f_sample * fs) / fs
tieeg = loadmat(f"../NonLinearityData/iEEG/iEEG_ID03_001h.mat")["EEG"]
sos = signal.butter(4, [1, 4], "bp", False, "sos", f_sample)
filtered = signal.resample(signal.sosfiltfilt(sos, tieeg[candi, :]).T, 3600 * fs).T
asas = np.argsort(np.argsort(filtered[:, 22000:22600], axis=1), axis=1)
fig, ax = plt.subplots(5, 4, figsize=(18, 18))
n = 0
for k in range(len(candi)):
    for l in range(k + 1, len(candi)):
        ax[n // 4, n % 4].hist2d(asas[k, :], asas[l, :], bins=9)
        ax[n // 4, n % 4].set_title(f"{candi[k]} - {candi[l]}")
        if candi[k] in inner and candi[l] in inner:
            ax[n // 4, n % 4].scatter([300], [300], marker="*", color="r", alpha=0.8)
        ax[n // 4, n % 4].set_xticks([])
        ax[n // 4, n % 4].set_yticks([])
        ax[n // 4, n % 4].set_aspect("1")
        # ax[n//5,n%5].plot([0,3600], [8,8], ":r")
        n += 1
        if n == 20:
            break
    if n == 20:
        break
plt.show()

In [ ]:
n